# Campus coffee sales: exponential smoothing forecast

> **Before you run:** choose **Copy to Drive** at the top of this page (or **File → Save a copy in Drive**). Your copy opens in a new tab. In your copy, choose **Runtime → Run all**. Your copy is yours, so Colab runs it without a warning, and it keeps your changes.

**Exponential Smoothing Method** · a worked example from the StatNex teaching guide

**The story.** A campus coffee shop counted cups sold every day for four weeks (28 days), starting on a Monday. Weekdays are busy and weekends quiet. The manager wants a quick forecast for tomorrow (day 29, a Monday) to plan how much milk to order.

**The question.** Is simple exponential smoothing a good way to forecast these sales?

If you run this page without a copy, Colab first warns that the notebook was not written by Google. It reads none of your data, so **Run anyway** is safe, but your changes are not kept.

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `08_time_series/exponential_smoothing.csv` (28 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
Day,Day of week,Cups sold
1.0,Monday,195.0
2.0,Tuesday,185.0
3.0,Wednesday,182.0
4.0,Thursday,191.0
5.0,Friday,188.0
6.0,Saturday,77.0
7.0,Sunday,76.0
8.0,Monday,187.0
9.0,Tuesday,197.0
10.0,Wednesday,175.0
11.0,Thursday,192.0
12.0,Friday,196.0
13.0,Saturday,112.0
14.0,Sunday,85.0
15.0,Monday,182.0
16.0,Tuesday,202.0
17.0,Wednesday,185.0
18.0,Thursday,207.0
19.0,Friday,194.0
20.0,Saturday,150.0
21.0,Sunday,87.0
22.0,Monday,204.0
23.0,Tuesday,199.0
24.0,Wednesday,202.0
25.0,Thursday,214.0
26.0,Friday,223.0
27.0,Saturday,139.0
28.0,Sunday,91.0

## Libraries

In [ ]:
import numpy as np
import pandas as pd
from statsmodels.tsa.holtwinters import SimpleExpSmoothing
from statsmodels.stats.diagnostic import acorr_ljungbox

pd.set_option("display.width", 120)

## The data

In [ ]:
# A time series in row order. The order of the rows is the data, so a series may not
# have blank cells: dropping one would make two non-adjacent periods neighbours.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
columns = ["Cups sold"]
# Each column is its own series.
series = {name: data[name] for name in columns}
alpha = 0.3
print(f"Smoothing constant: {alpha}")

## Exponential smoothing

In [ ]:
# Simple exponential smoothing: S(t) is the forecast for period t made from everything
# up to t - 1,
#     S(t + 1) = alpha * x(t) + (1 - alpha) * S(t),
# a weighted average in which an observation k periods old carries the weight
# alpha * (1 - alpha)^k: the newest data count most, and old data fade but never vanish.
# It assumes the series has a level that drifts, with no trend and no repeating season;
# on a series with a weekly pattern the forecast trails one step behind it.
# statsmodels' SimpleExpSmoothing does the recursion. alpha is fixed (optimized=False)
# rather than estimated, and initialization_method="known" sets S(1). Its fitted values
# are S(1) to S(n), and forecast(1) is S(n + 1), the forecast for the next period.
# The residual x(t) - S(t) is the one-step forecast error. Period 1's error is 0 by
# construction when S(1) is the first observation; it is still counted, as Stata's
# tssmooth counts it, so the mean squared error divides by n.
tables, fits = [], []
ljung_box = []
for name, values in series.items():
    x = values.to_numpy(float)
    n = len(x)
    # The starting value S(1) is the first observation.
    start = x[0]
    model = SimpleExpSmoothing(x, initialization_method="known", initial_level=start)
    ses = model.fit(smoothing_level=alpha, optimized=False)
    smoothed = np.append(ses.fittedvalues, ses.forecast(1))
    residual = x - smoothed[:n]
    rss = (residual ** 2).sum()
    table = pd.DataFrame({"No.": np.arange(1, n + 2), name: np.append(x, np.nan),
                          "Smoothed value": smoothed, "Residual": np.append(residual, np.nan)})
    fit = pd.Series({"Residual sum of squares": rss, "Mean residual sum of squares": rss / n,
                     "Root mean residual sum of squares": np.sqrt(rss / n)})
    tables.append(table)
    fits.append(fit)
    print()
    print(table.to_string(index=False))
    print(fit.to_string())
    # Ljung-Box test of the forecast errors: Q(h) = n(n + 2) * sum of r(k)^2 / (n - k)
    # over k = 1..h, where r(k) is the errors' autocorrelation at lag k, against
    # chi-square on h degrees of freedom, at every lag h up to min(10, n / 5) (Hyndman
    # and Athanasopoulos's rule). H0: the errors are uncorrelated with their own past.
    # A small P value says a pattern is left that the forecast could have used; it does
    # not say which. Read the row chosen in advance, normally the last. Alpha was given,
    # not estimated, so no parameter is taken off the degrees of freedom. A series
    # shorter than 5 has no lag to test.
    lags = min(10, n // 5)
    if lags >= 1:
        test = acorr_ljungbox(residual, lags=lags)
        ljung_box.append(pd.DataFrame({"lag": test.index, "Q": test["lb_stat"],
                                       "df": test.index, "P": test["lb_pvalue"]}))
        print(ljung_box[-1].to_string(index=False))

## What the result says

Simple exponential smoothing forecasts the next day from a weighted average of the past, where recent days count more. The smoothing constant, here 0.3, is the weight given to the newest day.

The forecast for day 29, a Monday, is about 155 cups. But the four Mondays so far sold 195, 187, 182 and 204 cups, so the forecast is far too low. The residuals (actual sales minus the smoothed value) show the reason: large negative values every weekend, such as -112 on day 6. The root mean residual sum of squares, a typical size of the miss, is about 50 cups. The Ljung-Box test at lag 5 has a P value of 0.012, so the errors still hold a pattern.

Simple exponential smoothing is not a good forecast here, because it cannot see the weekly pattern. Use a method that models the day of the week.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.